**Known issue:** despite the original filename, this notebook is a MobileNetV2 transfer-learning model, not AlexNet or ResNet. This run crashed partway through epoch 1 with a `CUDNN_STATUS_EXECUTION_FAILED` error (a GPU/cuDNN-level failure, not a code bug) and was never retried. No results exist from this notebook. See `prep/deepdive.md` for details.

In [4]:
import tensorflow as tf
import tensorflow_addons as tfa
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import LearningRateScheduler, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras import regularizers
from tensorflow.keras.mixed_precision import set_global_policy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Enable mixed precision training
set_global_policy('mixed_float16')

# Define dataset paths
train_data_dir = "E:/Projects/Strabismus/Final_Train_data"
val_data_dir = "E:/Projects/Strabismus/Split_Data/val"
test_data_dir = "E:/Projects/Strabismus/Split_Data/test"

# Image parameters
img_height, img_width = 227, 227
batch_size = 32  # Reduced batch size to avoid memory issues
num_classes = 5  

# Data Augmentation
train_datagen = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    val_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.arange(num_classes), y=train_generator.classes)
class_weight_dict = dict(enumerate(class_weights))

# Clear unused variables to free memory
tf.keras.backend.clear_session()
gc.collect()

# Define AlexNet-like model
def AlexNet(input_shape, num_classes):
    base_model = tf.keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = Dropout(0.5)(x)
    output = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=output)
    return model

# Instantiate model
model = AlexNet((img_height, img_width, 3), num_classes)

# Define learning rate scheduler
def lr_scheduler(epoch, lr):
    if epoch % 10 == 0 and epoch > 0:
        return lr * 0.5
    return lr

callback_lr = LearningRateScheduler(lr_scheduler)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.7, patience=7, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# Compile model
optimizer = tfa.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4)
model.compile(optimizer=optimizer, 
              loss='categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

# Train the model
history = model.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[callback_lr, reduce_lr, early_stopping]
)

# Function to plot learning curves
def plot_learning_curves(history):
    plt.figure(figsize=(12, 5))
    
    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Accuracy over Epochs')
    
    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss over Epochs')
    
    plt.show()

# Evaluate model on test data
y_pred_probs = model.predict(test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

# Print evaluation metrics
eval_results = f"Accuracy: {accuracy:.4f}\nPrecision: {precision:.4f}\nRecall: {recall:.4f}\nF1 Score: {f1:.4f}"
print(eval_results)

# Plot Confusion Matrix
def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

plot_confusion_matrix(y_true, y_pred, list(test_generator.class_indices.keys()))
plot_learning_curves(history)

print("Model training, evaluation, and testing complete.")

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU, compute capability 8.6
Found 3600 images belonging to 5 classes.
Found 76 images belonging to 5 classes.
Found 81 images belonging to 5 classes.
9406464/9406464 [==============================] - 1s 0us/step
Epoch 1/50
 41/113 [=========>....................] - ETA: 16s - loss: 1.5541 - accuracy: 0.4428 - auc: 0.7555 - precision: 0.5970 - recall: 0.2462

UnknownError: Graph execution error:

Detected at node 'gradient_tape/model/block_16_depthwise/depthwise/DepthwiseConv2dNativeBackpropFilter' defined at (most recent call last):
    File "E:\Anaconda\envs\matrix\lib\runpy.py", line 197, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "E:\Anaconda\envs\matrix\lib\runpy.py", line 87, in _run_code
      exec(code, run_globals)
    File "E:\Anaconda\envs\matrix\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "E:\Anaconda\envs\matrix\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
      app.start()
    File "E:\Anaconda\envs\matrix\lib\site-packages\ipykernel\kernelapp.py", line 739, in start
      self.io_loop.start()
    File "E:\Anaconda\envs\matrix\lib\site-packages\tornado\platform\asyncio.py", line 205, in start
      self.asyncio_loop.run_forever()
    File "E:\Anaconda\envs\matrix\lib\asyncio\base_events.py", li